# Head-to-head on WEATHER-5K: Atlas vs TimesFM and Chronos (ParadoxAI Lab)

Phase 1a of the plan: Atlas-5K 10M and v2c against TimesFM 2.5, TimesFM 3.0 (evaluation only), Chronos-2 (univariate and
multivariate) and Chronos-Bolt, on exactly the same 2022 validation windows, with paired tests. Rules and predictions are
in `atlas/README.md` ("Head-to-head on WEATHER-5K"), written before this ran. The 2023 test year is not used here.

**Run:** Runtime → Change runtime type → **A100** → Save → **Run all**. Results go to Drive `PWM-10M/h2h/`; a re-run
skips models already scored.

In [ ]:
# 1. Drive and GPU
from google.colab import drive
drive.mount('/content/drive')
import os, shutil, json
W = '/content/drive/MyDrive/PWM-10M'
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 2. Code and the opponents' packages
!rm -rf /content/pwm && git clone -q --depth 1 -b atlas-v0.1 https://github.com/paradoxailab/paradox-world-model.git /content/pwm
!cd /content/pwm && git log --oneline -1
!pip install -q timesfm chronos-forecasting 2>&1 | tail -2

In [ ]:
# 3. WEATHER-5K from the Drive copy made by the 10M run (or downloaded again if missing)
R = '/content/raw'; os.makedirs(R, exist_ok=True)
X, M = f'{W}/data/W5K_X.npy', f'{W}/data/W5K_META.npz'
if os.path.exists(f'{R}/W5K_X.npy'):
    print('dataset already here')
elif os.path.exists(X):
    shutil.copy(M, R); shutil.copy(X, R); print('copied from Drive')
else:
    !cd /content/pwm/atlas/delhi/tools && python -u fetch_weather5k.py --out /content/raw --tmp /content/tmp --workers 8
!ls -la /content/raw

In [ ]:
# 4. Score every model on the same 2022 windows (cheapest first; each result is saved as it finishes)
%cd /content/pwm/atlas/delhi
os.environ['PW_RAW'] = '/content/raw'; os.environ['ATLAS_OUT'] = W
A10 = f'{W}/out/w5k_atlas_10m_h24_g1_s600.pt'
!python -u score_tsfm_w5k.py --year val --atlas atlas10m={A10},v2c=/content/pwm/models/series/57_atlas-5k_v2c_weather5k_adopted.pt --models persist,chronos-bolt,chronos2,chronos2mv,tfm25,tfm3 2>&1 | grep --line-buffered -v "Warning\|warn(" | tee -a "{W}/h2h_val_log.txt" 

In [ ]:
# 5. Paired comparison (validation: evidence and targets, not the final claim)
!python paired_w5k.py --year val --ours atlas10m --vs persist,chronos-bolt,chronos2,chronos2mv,tfm25,tfm3 | tee -a "{W}/h2h_val_log.txt"
!python paired_w5k.py --year val --ours v2c --vs chronos2,tfm25,tfm3 | tee -a "{W}/h2h_val_log.txt" 